# Amazon ML Challenge 2026 — Scalable Candidate Generation
### Memory-Efficient, Modular, Multi-Stage Blocking Pipeline

This notebook is optimized for **Google Colab (CPU or GPU)**.
It uses **Polars** for sub-second dataset loading, caches intermediate artifacts (normalized columns, inverted indices, TF-IDF matrices), and allows executing individual stages independently without recomputation:

1. `PROFILE`: Dataset schema, row counts, country distributions
2. `BASELINE_BLOCKING`: Name exact, Country+Name, First-token prefix, Token overlap
3. `ADDRESS_BLOCKING`: Address & postal code blocking conditioned on country
4. `TFIDF_RETRIEVAL`: Scalable char n-gram TF-IDF retrieval partitioned by country
5. `MISS_ANALYSIS`: Diagnostic drill-down into missed true matches
6. `ABLATION`: Standalone & progressive union contribution analysis
7. `FINAL_CANDIDATE_GENERATION`: Candidate deduplication, metric reporting, and submission validation

In [ ]:
# ---------------------------------------------------------------------------
# Step 1: Dependencies & Repository Setup
# ---------------------------------------------------------------------------
!pip install -q polars pyarrow scikit-learn scipy numpy

import os
import sys
import subprocess
import importlib

# If running in Colab and repository is cloned or in Drive:
REPO_DIR = '/content/Amazon-ML-Challenge-2026'
REPO_URL = 'https://github.com/lakshya0101/Amazon-ML-Challenge-2026.git'
BRANCH = 'feature/aditya-blocking'

if os.path.exists('/content'):
    if not os.path.exists(REPO_DIR):
        print(f"Cloning repository branch '{BRANCH}'...")
        subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'checkout', BRANCH], check=True)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Verify branch and HEAD
branch_out = subprocess.run(['git', 'branch', '--show-current'], capture_output=True, text=True).stdout.strip()
head_out = subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout.strip()

print(f"Working directory: {os.getcwd()}")
print(f"Current branch:    {branch_out}")
print(f"Current HEAD:      {head_out}")

# Invalidate import caches and reload modules if already in sys.modules
importlib.invalidate_caches()
modules_to_reload = [
    'src.data.record',
    'src.data.loader',
    'src.config',
    'src.normalization.name_normalizer',
    'src.blocking.candidate_generator',
    'src.blocking.address_blocker',
    'src.blocking.tfidf_blocker',
    'src.blocking.miss_analysis',
    'src.blocking.ablation',
    'src.evaluation.blocking_metrics',
]
for mod_name in modules_to_reload:
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])


In [ ]:
# ---------------------------------------------------------------------------
# Step 2: Configurable Dataset & Artifact Paths
# ---------------------------------------------------------------------------
from src.config import Config

# Set dataset root according to your Colab environment (e.g. Google Drive or /content)
# If dataset is in Google Drive, uncomment:
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_ROOT = '/content/drive/MyDrive/Amazon_ML/student_resource/dataset'

DATASET_ROOT = os.getenv('AMAZON_ML_DATASET_DIR', 'student_resource/dataset')
OUTPUT_DIR = 'outputs'
CACHE_DIR = 'cache'

cfg = Config(
    dataset_root=DATASET_ROOT,
    output_dir=OUTPUT_DIR,
    cache_dir=CACHE_DIR,
    max_token_cap=2000,
    tfidf_top_k=20,
)

print(f"Dataset root:  {cfg.dataset_root}")
print(f"Train dir:     {cfg.train_dir}")
print(f"Cache dir:     {cfg.cache_dir}")
print(f"Outputs dir:   {cfg.output_dir}")

## Stage 1: PROFILE
Quickly inspect dataset size, column names, missing values, and country distributions.

In [ ]:
# Run dataset profiling script
!python3 src/profile_dataset.py

import json
with open('outputs/dataset_profile.json') as f:
    profile = json.load(f)
print("Profile summary:")
for fname, stats in profile['sources'].items():
    print(f"  {fname:<25} rows={stats['row_count']:>10,} missing_name={stats['missing_values']['business_name']}")

## Stage 2: Fast Data Loading & Inverted Index Caching
Loads S1, S2, S3 with column projections using Polars and caches indices.

In [ ]:
from src.data.loader import load_source_tuples, load_ground_truth_fast
import time

t0 = time.time()
print("Loading dataset...")
cols = ["entity_id", "business_name", "business_address", "country"]
s1_rows = load_source_tuples(cfg.train_source1, columns=cols)
s2_rows = load_source_tuples(cfg.train_source2, columns=cols)
s3_rows = load_source_tuples(cfg.train_source3, columns=cols)
ground_truth = load_ground_truth_fast(cfg.train_ground_truth)
all_s1_ids = [r[0] for r in s1_rows]

print(f"Loaded in {time.time()-t0:.1f}s: S1={len(s1_rows):,}, S2={len(s2_rows):,}, S3={len(s3_rows):,}, GT={len(ground_truth):,}")

## Stage 3: BASELINE_BLOCKING
Runs exact normalized name, country + name, first token prefix, and informative token overlap.

In [ ]:
from src.blocking.candidate_generator import (
    blocker_exact_norm_name,
    blocker_country_norm_name,
    blocker_first_token,
    blocker_token_overlap,
    union_candidates,
)
from src.evaluation.blocking_metrics import evaluate_blocking, print_report

baseline_blockers = []
print("Running B1: Exact normalized name...")
c1, n1 = blocker_exact_norm_name(s1_rows, s2_rows, s3_rows, cache_dir=cfg.cache_dir)
baseline_blockers.append((c1, n1))

print("Running B2: Country + normalized name...")
c2, n2 = blocker_country_norm_name(s1_rows, s2_rows, s3_rows, cache_dir=cfg.cache_dir)
baseline_blockers.append((c2, n2))

print("Running B3: First token prefix...")
c3, n3 = blocker_first_token(s1_rows, s2_rows, s3_rows, cache_dir=cfg.cache_dir)
baseline_blockers.append((c3, n3))

print("Running B4: Informative token overlap...")
c4, n4 = blocker_token_overlap(s1_rows, s2_rows, s3_rows, max_candidates_per_token=cfg.max_token_cap, cache_dir=cfg.cache_dir)
baseline_blockers.append((c4, n4))

merged_baseline, _ = union_candidates(baseline_blockers, all_s1_ids)
res_baseline = evaluate_blocking(merged_baseline, ground_truth)
print_report(res_baseline, title="Baseline Blocking (B1-B4)")

## Stage 4: ADDRESS_BLOCKING
Recovers entities with high name variation but identical address/postal codes conditioned on country.

In [ ]:
from src.blocking.address_blocker import blocker_country_postal_initial

print("Running Address Blocker...")
c_addr, n_addr = blocker_country_postal_initial(s1_rows, s2_rows, s3_rows)
res_addr = evaluate_blocking(c_addr, ground_truth)
print_report(res_addr, title="Address Postal Initial Blocker")

## Stage 5: TFIDF_RETRIEVAL
Sub-linear character 3-gram TF-IDF retrieval partitioned by country to capture typos and abbreviations.

In [ ]:
from src.blocking.tfidf_blocker import blocker_tfidf_retrieval

print("Running TF-IDF Retrieval...")
tfidf_cache = os.path.join(cfg.cache_dir, "tfidf_cache")
c_tfidf, n_tfidf = blocker_tfidf_retrieval(
    s1_rows, s2_rows, s3_rows,
    top_k=cfg.tfidf_top_k,
    cache_path=tfidf_cache
)
res_tfidf = evaluate_blocking(c_tfidf, ground_truth)
print_report(res_tfidf, title="TF-IDF Candidate Retrieval")

## Stage 6: MISS_ANALYSIS
Examine false negatives: which ground-truth pairs were missed and why?

In [ ]:
from src.blocking.miss_analysis import analyze_misses, save_miss_analysis

all_blockers = baseline_blockers + [(c_addr, n_addr)]
if c_tfidf:
    all_blockers.append((c_tfidf, n_tfidf))

final_cands, _ = union_candidates(all_blockers, all_s1_ids)

s1_dict = {r[0]: (r[1], r[2], r[3]) for r in s1_rows}
target_dict = {r[0]: (r[1], r[2], r[3]) for r in s2_rows}
for r in s3_rows:
    target_dict[r[0]] = (r[1], r[2], r[3])

miss_report = analyze_misses(final_cands, ground_truth, s1_dict, target_dict)
save_miss_analysis(miss_report, os.path.join(cfg.output_dir, "miss_analysis.json"))

print("Miss reasons breakdown:")
for reason, count in miss_report["miss_reasons"].items():
    print(f"  {reason:<30} {count:>8,}")

## Stage 7: ABLATION STUDY
Evaluates individual vs progressive union recall and pair volume.

In [ ]:
from src.blocking.ablation import run_ablation_study

ablation_results = run_ablation_study(
    all_blockers,
    ground_truth,
    all_s1_ids,
    out_path=os.path.join(cfg.output_dir, "ablation_study.json")
)

print("Progressive Union Recall:")
for stage, metrics in ablation_results["progressive_union"].items():
    print(f"  {stage:<45} Recall: {metrics['recall']:.4f} | Pairs: {metrics['total_candidate_pairs']:>10,} | Avg/S1: {metrics['avg_candidates_per_s1']:.1f}")

## Stage 8: FINAL_CANDIDATE_GENERATION & SUBMISSION VALIDATION
Saves `outputs/candidate_pairs.tsv` and checks formatting with `student_resource/utils/validate_submission.py`.

In [ ]:
from src.blocking.candidate_generator import write_candidate_pairs_tsv

cand_output_file = os.path.join(cfg.output_dir, "candidate_pairs.tsv")
write_candidate_pairs_tsv(final_cands, all_s1_ids, cand_output_file)
print(f"Successfully saved candidate pairs to: {cand_output_file}")

# Check formatting with validator
!python3 student_resource/utils/validate_submission.py \
    --candidate outputs/candidate_pairs.tsv \
    --test-dir student_resource/dataset/train